# Step 3 Solution Guide: Chunking, Embeddings, and Vector Index

Step 3 builds the retrieval memory for the product. The earlier pipeline can feel blurry when the corpus is thin, metadata is missing, or the embedding index is not saved carefully. This notebook keeps the 2020-through-latest filing scope attached to every chunk and gives both a recommended dense embedding path and a transparent TF-IDF fallback.


## 1. Path Setup


In [ ]:
from pathlib import Path
import pandas as pd

# In Google Colab, this is the expected project folder.
# If you run locally, replace this path with Path.cwd() / "project_sec10k_rag".
BASE_DIR = Path("/content/drive/MyDrive/project_sec10k_rag")

DATA_DIR = BASE_DIR / "data"
OUTPUTS_DIR = DATA_DIR / "outputs"
REPORTS_DIR = BASE_DIR / "reports"

for folder in [DATA_DIR, OUTPUTS_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", BASE_DIR)

CLEANED_ITEMS_DIR = DATA_DIR / "cleaned_items"
CHUNKED_DIR = DATA_DIR / "chunked"
EMBEDDINGS_DIR = DATA_DIR / "embeddings"
VECTOR_STORE_DIR = DATA_DIR / "vector_store"

for folder in [CHUNKED_DIR, EMBEDDINGS_DIR, VECTOR_STORE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


## 2. Chunk Text With Overlap

Chunks should be large enough to preserve meaning but small enough to retrieve precise evidence. This simple version uses words as a token approximation. If your team uses `tiktoken`, replace `text.split()` with model-specific tokens.


In [ ]:
def chunk_words(text, chunk_size=350, overlap=75):
    words = str(text).split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

example_chunks = chunk_words("risk " * 900, chunk_size=350, overlap=75)
print("Example chunk count:", len(example_chunks))


## 3. Build the Chunk Table

Each chunk needs metadata. Without metadata, the RAG answer cannot cite company, year, item, or NAICS scope.


In [ ]:
cleaned_metadata_path = OUTPUTS_DIR / "cleaned_item_metadata.csv"

if cleaned_metadata_path.exists():
    cleaned_metadata = pd.read_csv(cleaned_metadata_path)
else:
    cleaned_metadata = pd.DataFrame(columns=["cleaned_file"])

chunk_records = []

for _, row in cleaned_metadata.iterrows():
    cleaned_path = CLEANED_ITEMS_DIR / row["cleaned_file"]
    if not cleaned_path.exists():
        continue

    text = cleaned_path.read_text(encoding="utf-8", errors="ignore")
    chunks = chunk_words(text, chunk_size=450, overlap=100)

    for i, chunk_text in enumerate(chunks):
        chunk_records.append(
            {
                "chunk_id": f"{row.get('ticker', 'UNK')}_{row.get('filing_year', 'YEAR')}_item_{row.get('item_number', 'NA')}_chunk_{i:04d}",
                "source_file": row.get("source_file", ""),
                "extracted_file": row.get("extracted_file", ""),
                "cleaned_file": row["cleaned_file"],
                "naics_code": row.get("naics_code", ""),
                "company_name": row.get("company_name", ""),
                "ticker": row.get("ticker", ""),
                "filing_year": row.get("filing_year", ""),
                "filing_date": row.get("filing_date", ""),
                "item_number": row.get("item_number", ""),
                "token_count": len(chunk_text.split()),
                "text": chunk_text,
            }
        )

chunks_df = pd.DataFrame(chunk_records)
chunks_path = CHUNKED_DIR / "sec10k_chunks.parquet"

if not chunks_df.empty:
    chunks_df.to_parquet(chunks_path, index=False)
    chunk_summary = (
        chunks_df.groupby(["ticker", "item_number"], dropna=False)
        .agg(chunks=("chunk_id", "count"), median_tokens=("token_count", "median"))
        .reset_index()
    )
    chunk_summary.to_csv(OUTPUTS_DIR / "chunk_summary_step3.csv", index=False)
    print("Saved:", chunks_path)
    display(chunk_summary.head(20))
else:
    print("No chunks created yet. Confirm Step 1 extraction and Step 2 cleaning completed.")

chunks_df.head()


## 4. Create Embeddings and Save a Search Index

Embeddings turn each text chunk into a numeric representation for semantic search. The recommended path is:

1. Use a sentence embedding model to create dense vectors.
2. Normalize vectors so cosine-style similarity works well.
3. Save both the vectors and the chunk metadata.
4. Build a FAISS index when FAISS is available.

The cell below tries the recommended dense embedding path first. If `sentence-transformers` or `faiss` is unavailable, it falls back to a transparent TF-IDF baseline. The fallback is useful for teaching and debugging, but dense embeddings should usually retrieve better financial language once the corpus is large.


In [ ]:
import json
import numpy as np
import joblib
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

# In Colab, uncomment these if needed:
# !pip -q install sentence-transformers faiss-cpu

texts = chunks_df["text"].fillna("").tolist() if not chunks_df.empty else []
embedding_manifest = {
    "chunk_count": len(texts),
    "date_range": "2020 through latest available SEC filing at runtime",
    "recommended_backend": "sentence-transformers + FAISS",
}

backend = None

if not texts:
    print("No chunk text available. Finish Step 2 and the chunking cell first.")
else:
    try:
        from sentence_transformers import SentenceTransformer
        import faiss

        model_name = "sentence-transformers/all-MiniLM-L6-v2"
        model = SentenceTransformer(model_name)
        dense_embeddings = model.encode(
            texts,
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True,
        ).astype("float32")

        np.save(EMBEDDINGS_DIR / "dense_embeddings.npy", dense_embeddings)

        index = faiss.IndexFlatIP(dense_embeddings.shape[1])
        index.add(dense_embeddings)
        faiss.write_index(index, str(VECTOR_STORE_DIR / "faiss_index.idx"))

        backend = "sentence_transformers_faiss"
        embedding_manifest.update(
            {
                "backend_used": backend,
                "embedding_model": model_name,
                "embedding_shape": list(dense_embeddings.shape),
                "similarity": "inner product over normalized vectors",
            }
        )

    except Exception as exc:
        print("Dense embedding path unavailable; using TF-IDF fallback.")
        print("Reason:", exc)

        vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=1)
        tfidf_matrix = normalize(vectorizer.fit_transform(texts))

        sparse.save_npz(EMBEDDINGS_DIR / "tfidf_embeddings.npz", tfidf_matrix)
        joblib.dump(vectorizer, EMBEDDINGS_DIR / "tfidf_vectorizer.joblib")

        backend = "tfidf_sparse_baseline"
        embedding_manifest.update(
            {
                "backend_used": backend,
                "embedding_model": "TfidfVectorizer",
                "embedding_shape": list(tfidf_matrix.shape),
                "similarity": "dot product over normalized sparse vectors",
            }
        )

    chunks_df.drop(columns=["text"]).to_csv(VECTOR_STORE_DIR / "chunk_metadata.csv", index=False)
    (VECTOR_STORE_DIR / "embedding_manifest.json").write_text(
        json.dumps(embedding_manifest, indent=2),
        encoding="utf-8",
    )

    print(json.dumps(embedding_manifest, indent=2))


## 5. Retrieval Function and Audit Table

This function retrieves the most similar chunks for a question. The audit table is evidence that retrieval is working before generation begins.


In [ ]:
import json
import numpy as np
import joblib
from scipy import sparse
from sklearn.preprocessing import normalize

manifest_path = VECTOR_STORE_DIR / "embedding_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
backend = manifest.get("backend_used", "")

def retrieve(question, top_k=5):
    if chunks_df.empty:
        return pd.DataFrame()

    if backend == "sentence_transformers_faiss":
        from sentence_transformers import SentenceTransformer
        import faiss

        model = SentenceTransformer(manifest["embedding_model"])
        query_embedding = model.encode([question], normalize_embeddings=True).astype("float32")
        index = faiss.read_index(str(VECTOR_STORE_DIR / "faiss_index.idx"))
        scores, positions = index.search(query_embedding, top_k)
        results = chunks_df.iloc[positions[0]].copy()
        results["similarity_score"] = scores[0]
        results["retrieval_backend"] = backend
        return results

    vectorizer = joblib.load(EMBEDDINGS_DIR / "tfidf_vectorizer.joblib")
    matrix = sparse.load_npz(EMBEDDINGS_DIR / "tfidf_embeddings.npz")
    query_vector = normalize(vectorizer.transform([question]))
    scores = (matrix @ query_vector.T).toarray().ravel()
    positions = scores.argsort()[::-1][:top_k]
    results = chunks_df.iloc[positions].copy()
    results["similarity_score"] = scores[positions]
    results["retrieval_backend"] = backend or "tfidf_sparse_baseline"
    return results

test_questions = [
    "What cybersecurity risks do companies disclose?",
    "How do companies describe revenue growth drivers?",
    "What competition risks appear in the industry?",
    "How do companies discuss cloud infrastructure costs?",
    "What macroeconomic risks affect performance?",
]

audit_rows = []

for question in test_questions:
    hits = retrieve(question, top_k=5)
    for _, hit in hits.iterrows():
        audit_rows.append(
            {
                "question": question,
                "chunk_id": hit["chunk_id"],
                "ticker": hit.get("ticker", ""),
                "filing_year": hit.get("filing_year", ""),
                "item_number": hit.get("item_number", ""),
                "similarity_score": hit["similarity_score"],
                "retrieval_backend": hit["retrieval_backend"],
                "manual_judgment": "review",
                "evidence_preview": str(hit.get("text", ""))[:300],
            }
        )

retrieval_audit = pd.DataFrame(audit_rows)
retrieval_audit.to_csv(OUTPUTS_DIR / "retrieval_audit_step3.csv", index=False)
retrieval_audit.head(15)


## Step 3 Interpretation Prompt

After running retrieval, replace `manual_judgment = review` with `relevant`, `partially relevant`, or `not relevant`. Then write a short note explaining whether chunk size, overlap, or embedding choice should change.
